# §01 Convex Optimization — Exercises

**Chapter 8: Optimization** | [Notes](notes.md) | [Theory](theory.ipynb)

---

Eight graded exercises covering convex sets, convex functions, strong convexity, smoothness,
problem classes (LP/QP/SDP), duality, proximal operators, and an end-to-end ML application.

**Difficulty:** ★ Easy · ★★ Medium · ★★★ Hard

---


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
HAS_MPL = True
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
except ImportError:
    HAS_MPL = False

try:
    from scipy.optimize import linprog, minimize
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

np.random.seed(2024)

# ── Grading helpers ────────────────────────────────────────────────────────────
def header(title, stars):
    bar = "=" * 60
    print(f"{bar}\n  {title}  [{stars}]\n{bar}")

def check_close(name, got, want, tol=1e-4):
    ok = abs(got - want) < tol
    sym = "✓" if ok else "✗"
    print(f"  {sym}  {name}: got {got:.6g}, want {want:.6g}")
    return ok

def check_true(name, condition):
    sym = "✓" if condition else "✗"
    print(f"  {sym}  {name}: {condition}")
    return condition

def check_array(name, got, want, tol=1e-4):
    diff = np.max(np.abs(np.array(got) - np.array(want)))
    ok = diff < tol
    sym = "✓" if ok else "✗"
    print(f"  {sym}  {name}: max_diff={diff:.2e}")
    return ok

print("Setup complete. Helpers: header(), check_close(), check_true(), check_array()")


## Exercise 1 ★ — Convex Set Membership

**Task:** Implement `is_in_convex_hull(points, query)` that tests whether `query` is in the
convex hull of `points` by checking whether there exist non-negative weights $\lambda_i \geq 0$
with $\sum_i \lambda_i = 1$ and $\sum_i \lambda_i p_i = q$.

Then verify the **convex combination property**: for any two points in a convex set $\mathcal{C}$,
the line segment between them must also be in $\mathcal{C}$.

**Test cases:**
- Points: triangle with vertices $\{(0,0), (1,0), (0,1)\}$
- Query 1: $(0.2, 0.3)$ → should be **inside** (λ = [0.5, 0.2, 0.3])
- Query 2: $(0.6, 0.6)$ → should be **outside** (sum > 1)
- Query 3: $(0.5, 0.5)$ → should be **on boundary**

**Hint:** Use `scipy.optimize.linprog` to check feasibility of the LP:
$$\min 0 \quad \text{s.t.} \quad P\lambda = q, \; \mathbf{1}^\top \lambda = 1, \; \lambda \geq 0$$


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def is_in_convex_hull(points, query):
    # TODO: implement using scipy.optimize.linprog
    # points: (n_points, d) array
    # query:  (d,) array
    # Returns: bool
    raise NotImplementedError

# Test
triangle = np.array([[0., 0.], [1., 0.], [0., 1.]])
q1 = np.array([0.2, 0.3])
q2 = np.array([0.6, 0.6])
q3 = np.array([0.5, 0.5])

# Uncomment to test your implementation:
# print(is_in_convex_hull(triangle, q1))  # Expected: True
# print(is_in_convex_hull(triangle, q2))  # Expected: False
# print(is_in_convex_hull(triangle, q3))  # Expected: True


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def is_in_convex_hull(points, query):
    if not HAS_SCIPY:
        # Fallback: check via norm for 2D triangle only
        return True  # placeholder
    n, d = points.shape
    # Minimise 0 s.t. P @ lambda = q, sum(lambda) = 1, lambda >= 0
    c = np.zeros(n)
    # Equality: [P; 1^T] @ lambda = [q; 1]
    A_eq = np.vstack([points.T, np.ones((1, n))])
    b_eq = np.concatenate([query, [1.0]])
    bounds = [(0, None)] * n
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    return res.status == 0  # 0 = optimal solution found

header("Exercise 1: Convex Set Membership", "★")
check_true("q1=(0.2,0.3) inside triangle", is_in_convex_hull(triangle, q1))
check_true("q2=(0.6,0.6) outside triangle", not is_in_convex_hull(triangle, q2))
check_true("q3=(0.5,0.5) on boundary",  is_in_convex_hull(triangle, q3))

# Convex combination property
v1, v2 = triangle[0], triangle[1]
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    mid = alpha * v1 + (1 - alpha) * v2
    check_true(f"  midpoint α={alpha}: in hull", is_in_convex_hull(triangle, mid))


## Exercise 2 ★ — Convex Function Verification

**Task:** Write a function `check_convexity(f, x_range, n_tests)` that empirically verifies
convexity by sampling random pairs $(x, y)$ and $\alpha \in [0,1]$ and checking:
$$f(\alpha x + (1-\alpha) y) \leq \alpha f(x) + (1-\alpha) f(y)$$

Apply it to verify (or refute) convexity for:
1. $f(x) = x^2$ → convex
2. $f(x) = \log(1 + e^x)$ (softplus) → convex
3. $f(x) = -\log(x)$ on $(0, \infty)$ → convex
4. $f(x) = \sin(x)$ → **not** convex
5. $f(x) = |x|$ → convex

Report the maximum violation $\max_{x,y,\alpha} [f(\alpha x + (1-\alpha)y) - \alpha f(x) - (1-\alpha)f(y)]$
for each function. A function is empirically convex if the max violation is $\leq 10^{-9}$.


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def check_convexity(f, x_range=(-5, 5), n_tests=2000, tol=1e-9):
    # TODO: sample n_tests random triples (x, y, alpha) and check
    # Returns (is_convex: bool, max_violation: float)
    raise NotImplementedError

functions = {
    "x^2":           lambda x: x**2,
    "softplus":      lambda x: np.log1p(np.exp(x)),
    "-log(x)":       lambda x: -np.log(np.maximum(x, 1e-10)),
    "sin(x)":        lambda x: np.sin(x),
    "|x|":           lambda x: np.abs(x),
}

# Uncomment to test:
# for name, f in functions.items():
#     is_cvx, viol = check_convexity(f)
#     print(f"{name:15}: convex={is_cvx}, max_violation={viol:.2e}")


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def check_convexity(f, x_range=(-5, 5), n_tests=2000, tol=1e-9):
    xs = np.random.uniform(x_range[0], x_range[1], n_tests)
    ys = np.random.uniform(x_range[0], x_range[1], n_tests)
    alphas = np.random.uniform(0, 1, n_tests)
    lhs = f(alphas * xs + (1 - alphas) * ys)
    rhs = alphas * f(xs) + (1 - alphas) * f(ys)
    violations = lhs - rhs
    max_viol = np.max(violations)
    return max_viol <= tol, max_viol

header("Exercise 2: Convex Function Verification", "★")
expected = {"x^2": True, "softplus": True, "-log(x)": True, "sin(x)": False, "|x|": True}
for name, f in functions.items():
    is_cvx, viol = check_convexity(f, x_range=(0.1, 5) if name == "-log(x)" else (-5, 5))
    check_true(f"{name:15}: convex={is_cvx} (expect {expected[name]}), viol={viol:.1e}",
               is_cvx == expected[name])


## Exercise 3 ★★ — Strong Convexity and Condition Number

Consider the quadratic $f(x) = \frac{1}{2} x^\top Q x$ where $Q \in \mathbb{R}^{n \times n}$ is symmetric PD.

**Task A:** Write `compute_params(Q)` that returns $(\mu, L, \kappa)$:
- $\mu = \lambda_{\min}(Q)$ (strong convexity modulus)
- $L = \lambda_{\max}(Q)$ (smoothness constant)
- $\kappa = L/\mu$ (condition number)

**Task B:** Verify the **descent lemma** numerically: for $f(x) = \frac{1}{2}\|Ax\|^2$,
show that $f(y) \leq f(x) + \nabla f(x)^\top(y-x) + \frac{L}{2}\|y-x\|^2$ holds
for 1000 random pairs $(x, y)$.

**Task C:** Run gradient descent on three quadratics with $\kappa \in \{2, 20, 200\}$
starting from $x_0 = \mathbf{1}$ and step size $\eta = 1/L$.
Plot (or print) the number of iterations to reach $f(x_t) \leq 10^{-6} f(x_0)$.
Compare with the theoretical bound $t^* = \kappa \log(f_0/\epsilon)$.


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def compute_params(Q):
    # TODO: return (mu, L, kappa)
    raise NotImplementedError

def verify_descent_lemma(A, n_tests=1000):
    # TODO: return (all_satisfied: bool, max_violation: float)
    raise NotImplementedError

def gd_iterations(Q, eps_ratio=1e-6):
    # TODO: run GD from x0=ones, step=1/L, return (iterations, final_loss)
    raise NotImplementedError


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def compute_params(Q):
    eigs = np.linalg.eigvalsh(Q)
    mu = float(eigs.min())
    L = float(eigs.max())
    kappa = L / mu
    return mu, L, kappa

def verify_descent_lemma(A, n_tests=1000):
    Q = A.T @ A
    L = np.linalg.eigvalsh(Q).max()
    f  = lambda x: 0.5 * x @ Q @ x
    grad_f = lambda x: Q @ x
    n = A.shape[1]
    xs = np.random.randn(n_tests, n)
    ys = np.random.randn(n_tests, n)
    lhs = np.array([f(y) for y in ys])
    rhs = np.array([f(x) + grad_f(x) @ (y - x) + 0.5 * L * np.sum((y-x)**2)
                    for x, y in zip(xs, ys)])
    violations = lhs - rhs
    return np.all(violations <= 1e-10), float(violations.max())

def gd_iterations(Q, eps_ratio=1e-6):
    mu, L, kappa = compute_params(Q)
    eta = 1.0 / L
    x = np.ones(Q.shape[0])
    f = lambda v: 0.5 * v @ Q @ v
    f0 = f(x)
    target = eps_ratio * f0
    for t in range(1, 100001):
        x = x - eta * (Q @ x)
        if f(x) <= target:
            return t, f(x)
    return 100000, f(x)

header("Exercise 3: Strong Convexity and Condition Number", "★★")

# Task A
np.random.seed(0)
for kappa_target in [2, 20, 200]:
    n = 5
    # Build Q with desired condition number
    eigs = np.linspace(1.0, float(kappa_target), n)
    U, _ = np.linalg.qr(np.random.randn(n, n))
    Q = U @ np.diag(eigs) @ U.T
    Q = 0.5 * (Q + Q.T)  # symmetrise
    mu, L, kappa = compute_params(Q)
    check_close(f"κ={kappa_target} → computed κ", kappa, float(kappa_target), tol=0.5)

# Task B
A = np.random.randn(10, 5)
ok, viol = verify_descent_lemma(A)
check_true("Descent lemma satisfied (1000 tests)", ok)
print(f"  Max violation: {viol:.2e}")

# Task C — convergence iterations
print("\nTask C: Iterations to 1e-6 reduction")
for kappa_target in [2, 20, 200]:
    n = 10
    eigs = np.linspace(1.0, float(kappa_target), n)
    U, _ = np.linalg.qr(np.random.randn(n, n))
    Q = U @ np.diag(eigs) @ U.T; Q = 0.5*(Q+Q.T)
    mu, L, kappa = compute_params(Q)
    iters, floss = gd_iterations(Q)
    theory_bound = int(kappa * np.log(1.0 / 1e-6))
    print(f"  κ≈{kappa:.0f}: actual={iters}, theory≤{theory_bound}")
    check_true(f"  κ≈{kappa:.0f}: converged", floss <= 1e-6 * 0.5 * np.linalg.norm(np.ones(n))**2)


## Exercise 4 ★★ — LP Feasibility and Duality

**Background:** The LP duality theorem states that for a primal LP:
$$p^* = \min_{x} c^\top x \quad \text{s.t.} \quad Ax = b, \; x \geq 0$$

The dual is:
$$d^* = \max_{y} b^\top y \quad \text{s.t.} \quad A^\top y \leq c$$

**Strong duality**: $p^* = d^*$ when both are feasible (LP strong duality always holds by Farkas' lemma).

**Task A:** Solve a random LP instance with $m=5$ constraints and $n=10$ variables.
Verify $p^* = d^*$ (strong duality) to within $10^{-4}$.

**Task B:** Implement a simple 1D LP using the simplex method structure:
Minimise $c^\top x$ s.t. $Ax \leq b$, $x \geq 0$ using `scipy.optimize.linprog`.

**Task C:** Demonstrate **weak duality** by constructing a dual feasible point $y$ and
showing $b^\top y \leq p^*$ (dual objective ≤ primal optimal).


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def solve_primal_lp(c, A_eq, b_eq):
    # TODO: solve min c^T x s.t. A_eq x = b_eq, x >= 0
    # Return (optimal_value, x_opt) or (None, None) if infeasible
    raise NotImplementedError

def solve_dual_lp(c, A_eq, b_eq):
    # TODO: solve max b^T y s.t. A_eq^T y <= c
    # Return (optimal_value, y_opt) or (None, None) if unbounded
    raise NotImplementedError


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def solve_primal_lp(c, A_eq, b_eq):
    if not HAS_SCIPY:
        return None, None
    n = len(c)
    bounds = [(0, None)] * n
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if res.status == 0:
        return float(res.fun), res.x
    return None, None

def solve_dual_lp(c, A_eq, b_eq):
    if not HAS_SCIPY:
        return None, None
    m = len(b_eq)
    # max b^T y  s.t. A^T y <= c  ↔  min -b^T y  s.t. A^T y <= c
    # linprog: A_ub @ y <= b_ub
    res = linprog(-b_eq, A_ub=A_eq.T, b_ub=c, bounds=[(None, None)] * m, method='highs')
    if res.status == 0:
        return float(-res.fun), res.x
    return None, None

header("Exercise 4: LP Feasibility and Duality", "★★")

np.random.seed(42)
m, n = 5, 10
# Construct a feasible LP: generate x_true >= 0, set b = A x_true
A_eq = np.random.randn(m, n)
x_true = np.abs(np.random.randn(n)) + 0.1  # strictly positive
b_eq = A_eq @ x_true
c = np.random.randn(n)

p_star, x_opt = solve_primal_lp(c, A_eq, b_eq)
d_star, y_opt = solve_dual_lp(c, A_eq, b_eq)

if p_star is not None and d_star is not None:
    print(f"Primal optimal: p* = {p_star:.6f}")
    print(f"Dual optimal:   d* = {d_star:.6f}")
    check_close("Strong duality p* = d*", p_star, d_star, tol=1e-3)
    check_true("Primal feasible: A x = b", np.allclose(A_eq @ x_opt, b_eq, atol=1e-5))
    check_true("Primal feasible: x >= 0", np.all(x_opt >= -1e-8))

    # Task C: Weak duality with a dual feasible point
    # Any dual feasible y satisfies A^T y <= c → b^T y <= p*
    y_feasible = np.zeros(m)  # y=0 is dual feasible (A^T 0 = 0 <= c if c >= 0)
    # Use a small perturbation of y_opt toward zero
    y_weak = y_opt * 0.5
    dual_val_weak = float(b_eq @ y_weak)
    print(f"\nWeak duality: b^T y_weak = {dual_val_weak:.4f} vs p* = {p_star:.4f}")
    check_true("Weak duality: d(y_weak) <= p*", dual_val_weak <= p_star + 1e-6)
else:
    print("SciPy not available; skipping LP solve")


## Exercise 5 ★★ — Proximal Operator and ISTA

The **proximal gradient algorithm** (ISTA) solves:
$$\min_x f(x) + g(x)$$
where $f$ is smooth (gradient available) and $g$ is non-smooth (proximal operator available).

**Algorithm:** $x_{t+1} = \text{prox}_{\eta g}(x_t - \eta \nabla f(x_t))$

**Task A:** Implement `prox_l1(v, threshold)` — the soft-thresholding operator, proximal map of $\lambda\|\cdot\|_1$.

**Task B:** Implement `ista(A, b, lam, n_iter)` solving the Lasso:
$$\min_x \frac{1}{2}\|Ax - b\|^2 + \lambda\|x\|_1$$

Use step size $\eta = 1/L$ where $L = \|A\|_2^2$.

**Task C:** Compare ISTA vs gradient descent on the smooth part only (no L1).
Show that ISTA with L1 achieves a sparser solution with similar or better objective.

**Theoretical rate:** ISTA converges at $O(1/t)$ for convex $f+g$.


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def prox_l1(v, threshold):
    # TODO: element-wise soft-thresholding
    # prox_{t * lambda * ||.||_1}(v) = sign(v) * max(|v| - threshold, 0)
    raise NotImplementedError

def ista(A, b, lam, n_iter=500):
    # TODO: implement ISTA for Lasso
    # Returns (x_opt, losses) where losses[t] = 0.5*||Ax-b||^2 + lam*||x||_1
    raise NotImplementedError


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def prox_l1(v, threshold):
    return np.sign(v) * np.maximum(np.abs(v) - threshold, 0)

def ista(A, b, lam, n_iter=500):
    m, n = A.shape
    L = np.linalg.norm(A, 2)**2
    eta = 1.0 / L
    x = np.zeros(n)
    losses = []
    for _ in range(n_iter):
        grad = A.T @ (A @ x - b)
        x = prox_l1(x - eta * grad, eta * lam)
        loss = 0.5 * np.sum((A @ x - b)**2) + lam * np.sum(np.abs(x))
        losses.append(loss)
    return x, losses

# Setup
np.random.seed(7)
m, n = 50, 100
A_ex = np.random.randn(m, n)
x_true = np.zeros(n); x_true[:5] = np.random.randn(5)  # 5-sparse
b_ex = A_ex @ x_true + 0.01 * np.random.randn(m)
lam = 0.05

header("Exercise 5: ISTA / Proximal Gradient", "★★")

# Task A: prox_l1
v_test = np.array([-2.0, -0.3, 0.0, 0.5, 1.8])
st = prox_l1(v_test, 0.5)
expected_st = np.array([-1.5, 0.0, 0.0, 0.0, 1.3])
check_array("prox_l1([-2,-0.3,0,0.5,1.8], 0.5)", st, expected_st)

# Task B: ISTA
x_opt, losses = ista(A_ex, b_ex, lam=lam, n_iter=500)
sparsity = np.sum(np.abs(x_opt) > 1e-3)
print(f"\nISTA result:")
print(f"  Final loss:  {losses[-1]:.6f}")
print(f"  ||x||_0:     {sparsity} (true support: 5)")
print(f"  ||x-x_true||:{np.linalg.norm(x_opt - x_true):.4f}")
check_true("Sparsity ≤ 15 (recovers approximate support)", sparsity <= 15)
check_true("Loss decreased monotonically", all(losses[i] >= losses[i+1] - 1e-10
                                               for i in range(0, len(losses)-1, 10)))

# Task C: compare with plain GD (no L1)
x_gd = np.zeros(n); L_gd = np.linalg.norm(A_ex, 2)**2; eta_gd = 1.0/L_gd
losses_gd = []
for _ in range(500):
    x_gd = x_gd - eta_gd * (A_ex.T @ (A_ex @ x_gd - b_ex))
    losses_gd.append(0.5*np.sum((A_ex @ x_gd - b_ex)**2) + lam*np.sum(np.abs(x_gd)))

sparsity_gd = np.sum(np.abs(x_gd) > 1e-3)
print(f"\nGD (no prox) result:")
print(f"  Final loss:  {losses_gd[-1]:.6f}")
print(f"  ||x||_0:     {sparsity_gd} (dense)")
check_true("ISTA sparser than plain GD", sparsity < sparsity_gd)


## Exercise 6 ★★ — Fenchel Conjugate and Young's Inequality

**Task A:** Analytically derive (and numerically verify) the Fenchel conjugate for:
1. $f(x) = \frac{p}{q}|x|^q$ for $q > 1$ — conjugate is $f^*(y) = \frac{1}{p}|y|^p$ where $\frac{1}{p} + \frac{1}{q} = 1$
2. $f(x) = \log(1 + e^x)$ (softplus) — conjugate is binary entropy $H_b(y) = -y\log y - (1-y)\log(1-y)$

**Task B:** Verify **Young's inequality** $x^\top y \leq f(x) + f^*(y)$ for both functions
using 1000 random pairs $(x, y)$.

**Task C:** Show that equality in Young's inequality holds when $y \in \partial f(x)$:
For $f(x) = \frac{1}{2}x^2$, equality holds when $y = x$ (since $f'(x) = x$).

**Hint for Task A(2):** The softplus conjugate can be derived by solving $\max_x \{xy - \log(1+e^x)\}$.
Setting the derivative to zero: $y - \frac{e^x}{1+e^x} = 0 \Rightarrow e^x = \frac{y}{1-y}$.


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def softplus(x):
    return np.log1p(np.exp(x))

def softplus_conjugate(y):
    # TODO: binary entropy H_b(y) = -y*log(y) - (1-y)*log(1-y)
    # Valid for y in (0, 1)
    raise NotImplementedError

def verify_young(f, f_star, x_range, y_range, n=1000):
    # TODO: verify x*y <= f(x) + f*(y) for random x, y
    # Return (all_ok: bool, max_violation: float)
    raise NotImplementedError


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def softplus_conjugate(y):
    # H_b(y) = -y log y - (1-y) log(1-y)
    y = np.clip(y, 1e-10, 1 - 1e-10)
    return -y * np.log(y) - (1 - y) * np.log(1 - y)

def verify_young(f, f_star, x_range, y_range, n=1000):
    xs = np.random.uniform(x_range[0], x_range[1], n)
    ys = np.random.uniform(y_range[0], y_range[1], n)
    lhs = xs * ys
    rhs = f(xs) + f_star(ys)
    violations = lhs - rhs
    return bool(np.all(violations <= 1e-9)), float(violations.max())

header("Exercise 6: Fenchel Conjugate and Young's Inequality", "★★")

# Task A(1): q=2, p=2: f(x)=x^2/2, f*(y)=y^2/2
f_l2 = lambda x: 0.5 * x**2
fstar_l2 = lambda y: 0.5 * y**2
ys = np.linspace(-3, 3, 100)
xs = np.linspace(-3, 3, 1000)
fstar_num = np.array([np.max(y * xs - f_l2(xs)) for y in ys])
fstar_ana = fstar_l2(ys)
err = np.max(np.abs(fstar_num - fstar_ana))
check_close("Conjugate of 0.5x²: max_err", err, 0.0, tol=1e-3)

# Task A(2): softplus conjugate
ys_sp = np.linspace(0.05, 0.95, 100)
xs_sp = np.linspace(-10, 10, 2000)
fstar_sp_num = np.array([np.max(y * xs_sp - softplus(xs_sp)) for y in ys_sp])
fstar_sp_ana = softplus_conjugate(ys_sp)
err_sp = np.max(np.abs(fstar_sp_num - fstar_sp_ana))
check_close("Conjugate of softplus: max_err", err_sp, 0.0, tol=1e-3)

# Task B: Young's inequality
ok_l2, viol_l2 = verify_young(f_l2, fstar_l2, (-3, 3), (-3, 3))
ok_sp, viol_sp = verify_young(softplus, softplus_conjugate, (-5, 5), (0.05, 0.95))
check_true(f"Young's ineq (0.5x²): ok={ok_l2}, max_viol={viol_l2:.1e}", ok_l2)
check_true(f"Young's ineq (softplus): ok={ok_sp}, max_viol={viol_sp:.1e}", ok_sp)

# Task C: equality when y = f'(x) = x
print("\nTask C: Equality in Young's at y = ∇f(x)")
for x_val in [-2.0, 0.0, 1.5]:
    y_val = x_val  # f'(x) = x for 0.5 x^2
    xy = x_val * y_val
    fxfstary = f_l2(x_val) + fstar_l2(y_val)
    check_close(f"  x={x_val}: xy={xy:.2f} vs f(x)+f*(y)={fxfstary:.2f}", xy, fxfstary, tol=1e-9)


## Exercise 7 ★★★ — SVM Dual and KKT Conditions

The **hard-margin SVM** primal is:
$$\min_{w,b} \frac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w^\top x_i + b) \geq 1 \; \forall i$$

The Lagrangian is:
$$\mathcal{L}(w, b, \alpha) = \frac{1}{2}\|w\|^2 - \sum_i \alpha_i[y_i(w^\top x_i + b) - 1]$$

Minimising over $(w, b)$ gives the dual:
$$\max_{\alpha \geq 0} \sum_i \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j x_i^\top x_j \quad \text{s.t.} \quad \sum_i \alpha_i y_i = 0$$

**Task A:** Implement `svm_dual(X, y)` that solves the SVM dual using `scipy.optimize.minimize`
(or `linprog` for the QP form).

**Task B:** Recover the primal variables: $w^* = \sum_i \alpha_i^* y_i x_i$ and
$b^* = y_j - w^{*\top} x_j$ for any support vector $j$ (where $\alpha_j > 0$).

**Task C:** Verify the **KKT conditions**:
1. Stationarity: $w = \sum_i \alpha_i y_i x_i$
2. Primal feasibility: $y_i(w^\top x_i + b) \geq 1$
3. Dual feasibility: $\alpha_i \geq 0$
4. Complementary slackness: $\alpha_i[y_i(w^\top x_i + b) - 1] = 0$


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def svm_dual(X, y):
    # TODO: solve the SVM dual, return (alpha, w, b)
    # alpha: (n,) dual variables
    # w, b: primal solution recovered from alpha
    raise NotImplementedError

def check_kkt(X, y, alpha, w, b, tol=1e-4):
    # TODO: verify all 4 KKT conditions, return dict of results
    raise NotImplementedError


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def svm_dual(X, y):
    if not HAS_SCIPY:
        return None, None, None
    n = len(y)
    # Gram matrix with labels
    K = np.outer(y, y) * (X @ X.T)

    def neg_dual(alpha):
        return -(np.sum(alpha) - 0.5 * alpha @ K @ alpha)

    def neg_dual_grad(alpha):
        return -(np.ones(n) - K @ alpha)

    # Constraints: sum(alpha * y) = 0
    constraints = [{'type': 'eq', 'fun': lambda a: np.dot(a, y),
                    'jac': lambda a: y}]
    bounds = [(0, None)] * n
    alpha0 = np.ones(n) / n

    res = minimize(neg_dual, alpha0, jac=neg_dual_grad,
                   method='SLSQP', bounds=bounds, constraints=constraints,
                   options={'ftol': 1e-10, 'maxiter': 1000})

    alpha = np.maximum(res.x, 0)
    # Recover primal
    w = (alpha * y) @ X  # w = sum_i alpha_i y_i x_i
    sv_mask = alpha > 1e-4
    if sv_mask.any():
        sv_idx = np.where(sv_mask)[0][0]
        b_val = float(y[sv_idx] - w @ X[sv_idx])
    else:
        b_val = 0.0
    return alpha, w, b_val

def check_kkt(X, y, alpha, w, b, tol=1e-4):
    n = len(y)
    margins = y * (X @ w + b)
    cs = alpha * (margins - 1)
    return {
        'stationarity': np.allclose(w, (alpha * y) @ X, atol=tol),
        'primal_feasible': np.all(margins >= 1 - tol),
        'dual_feasible': np.all(alpha >= -tol),
        'complementary_slackness': np.all(np.abs(cs) < tol),
    }

header("Exercise 7: SVM Dual and KKT", "★★★")

# Generate linearly separable data
np.random.seed(3)
X_pos = np.random.randn(10, 2) + np.array([2., 0.])
X_neg = np.random.randn(10, 2) + np.array([-2., 0.])
X_svm = np.vstack([X_pos, X_neg])
y_svm = np.array([1.]*10 + [-1.]*10)

alpha, w, b = svm_dual(X_svm, y_svm)

if alpha is not None:
    print(f"Dual optimal value: {np.sum(alpha) - 0.5*alpha @ (np.outer(y_svm,y_svm)*(X_svm@X_svm.T)) @ alpha:.4f}")
    print(f"Primal ||w||²/2:    {0.5*np.dot(w,w):.4f}")
    n_sv = np.sum(alpha > 1e-4)
    print(f"Support vectors:    {n_sv}")

    kkt = check_kkt(X_svm, y_svm, alpha, w, b)
    for cond, ok in kkt.items():
        check_true(f"KKT {cond}", ok)

    # Strong duality check
    dual_val = np.sum(alpha) - 0.5*alpha @ (np.outer(y_svm,y_svm)*(X_svm@X_svm.T)) @ alpha
    primal_val = 0.5 * np.dot(w, w)
    check_close("Strong duality: dual = primal", dual_val, primal_val, tol=1e-3)
else:
    print("SciPy not available; skipping SVM solve")


## Exercise 8 ★★★ — End-to-End: Elastic Net Regularisation

**Elastic net** combines L1 and L2 regularisation:
$$\mathcal{L}(\theta) = \frac{1}{2n}\|X\theta - y\|^2 + \lambda_1\|\theta\|_1 + \frac{\lambda_2}{2}\|\theta\|^2$$

This is strongly convex (from the L2 term) and non-smooth (from the L1 term).

**Proximal operator of elastic net:**
$$\text{prox}_{\eta(\lambda_1\|\cdot\|_1 + \frac{\lambda_2}{2}\|\cdot\|^2)}(v) = \frac{\text{sign}(v) \cdot \max(|v| - \eta\lambda_1, 0)}{1 + \eta\lambda_2}$$

**Task A:** Implement `prox_elastic_net(v, eta, lam1, lam2)`.

**Task B:** Implement `proximal_gd_elastic_net(X, y, lam1, lam2, n_iter)` using proximal gradient.

**Task C:** Run on a sparse regression problem ($n=100, p=200$, 10 true coefficients).
Vary $\lambda_1 \in \{0.001, 0.01, 0.1\}$ with fixed $\lambda_2 = 0.01$ and report:
- Number of non-zeros in $\hat{\theta}$
- $\|\hat{\theta} - \theta^*\|_2$ (recovery error)
- Final objective value

**Task D:** Show that setting $\lambda_2 \to 0$ recovers Lasso, and $\lambda_1 \to 0$ recovers Ridge.
Verify that the elastic net objective lies between both extremes for the same $\lambda$ budget.


In [ ]:
# ── Scaffold ─────────────────────────────────────────────────────────────────
def prox_elastic_net(v, eta, lam1, lam2):
    # TODO: proximal operator of eta*(lam1*||.||_1 + lam2/2*||.||^2)
    raise NotImplementedError

def proximal_gd_elastic_net(X, y, lam1, lam2, n_iter=500):
    # TODO: proximal gradient descent for elastic net
    # Returns (theta_opt, losses)
    raise NotImplementedError


In [ ]:
# ── Solution ─────────────────────────────────────────────────────────────────
def prox_elastic_net(v, eta, lam1, lam2):
    # Soft-threshold then shrink
    st = np.sign(v) * np.maximum(np.abs(v) - eta * lam1, 0)
    return st / (1 + eta * lam2)

def proximal_gd_elastic_net(X, y, lam1, lam2, n_iter=500):
    n, p = X.shape
    L = np.linalg.norm(X, 2)**2 / n + lam2  # L-smoothness constant
    eta = 1.0 / L
    theta = np.zeros(p)
    losses = []
    for _ in range(n_iter):
        grad = X.T @ (X @ theta - y) / n
        theta = prox_elastic_net(theta - eta * grad, eta, lam1, lam2)
        loss = (0.5/n) * np.sum((X @ theta - y)**2) + lam1*np.sum(np.abs(theta)) + 0.5*lam2*np.dot(theta,theta)
        losses.append(loss)
    return theta, losses

header("Exercise 8: Elastic Net Regularisation", "★★★")

# Task A
v_test = np.array([-3.0, -0.5, 0.0, 0.8, 2.5])
prox_en = prox_elastic_net(v_test, eta=1.0, lam1=0.5, lam2=0.2)
# Manual: soft-thresh then shrink: st=[-2.5, 0, 0, 0.3, 2.0] / (1+0.2)
st_check = np.sign(v_test)*np.maximum(np.abs(v_test)-0.5,0) / 1.2
check_array("prox_elastic_net correctness", prox_en, st_check)

# Task B & C: sparse regression
np.random.seed(9)
n, p = 100, 200
X_en = np.random.randn(n, p) / np.sqrt(n)
theta_true = np.zeros(p); theta_true[:10] = np.random.randn(10)
y_en = X_en @ theta_true + 0.05 * np.random.randn(n)

lam2 = 0.01
print("\nTask C: Elastic Net vs Lambda_1")
print(f"{'lam1':>8} | {'nnz':>6} | {'||err||':>10} | {'loss':>12}")
print("-" * 46)
for lam1 in [0.001, 0.01, 0.1]:
    theta_hat, _ = proximal_gd_elastic_net(X_en, y_en, lam1=lam1, lam2=lam2)
    nnz = np.sum(np.abs(theta_hat) > 1e-3)
    err = np.linalg.norm(theta_hat - theta_true)
    loss = (0.5/n)*np.sum((X_en @ theta_hat - y_en)**2) + lam1*np.sum(np.abs(theta_hat)) + 0.5*lam2*np.dot(theta_hat,theta_hat)
    print(f"{lam1:>8.3f} | {nnz:>6d} | {err:>10.4f} | {loss:>12.6f}")
    check_true(f"  lam1={lam1}: nnz <= p", nnz <= p)

# Task D: Limiting cases
print("\nTask D: Limiting Cases")
# Lasso: lam2 -> 0
theta_lasso, _ = proximal_gd_elastic_net(X_en, y_en, lam1=0.01, lam2=1e-8)
nnz_lasso = np.sum(np.abs(theta_lasso) > 1e-3)

# Ridge (via L2 only): use lam1=0
theta_ridge, _ = proximal_gd_elastic_net(X_en, y_en, lam1=1e-8, lam2=0.01)
nnz_ridge = np.sum(np.abs(theta_ridge) > 1e-3)

theta_en, _ = proximal_gd_elastic_net(X_en, y_en, lam1=0.01, lam2=0.01)
nnz_en = np.sum(np.abs(theta_en) > 1e-3)

print(f"  Lasso  (lam2→0): nnz={nnz_lasso}")
print(f"  Elastic (equal): nnz={nnz_en}")
print(f"  Ridge  (lam1→0): nnz={nnz_ridge}")
check_true("Lasso sparser than Ridge", nnz_lasso <= nnz_ridge)


## Summary

Congratulations on completing the §01 Convex Optimization exercises!

### Skills Demonstrated
| Exercise | Topic | Key Result |
|---|---|---|
| 1 ★ | Convex set membership | LP feasibility for convex hull test |
| 2 ★ | Convex function verification | Empirical midpoint rule check |
| 3 ★★ | Strong convexity & condition number | μ, L, κ from eigenvalues; descent lemma |
| 4 ★★ | LP duality | Strong duality p* = d* numerically |
| 5 ★★ | Proximal gradient (ISTA) | Lasso recovery with soft-threshold |
| 6 ★★ | Fenchel conjugate | Young's inequality; softplus conjugate |
| 7 ★★★ | SVM dual & KKT | Dual SVM; KKT complementary slackness |
| 8 ★★★ | Elastic net | Proximal operator composition; sparsity |

### What's Next
- **§02 Gradient Descent**: convergence proofs using the μ, L, κ parameters from Exercise 3
- **§04 Constrained Optimization**: the KKT conditions from Exercise 7 in full generality
- **§05 Stochastic Optimization**: Exercise 5's ISTA → FISTA → stochastic proximal gradient

---
*[← Notes](notes.md) | [Theory Notebook](theory.ipynb) | [§02 Gradient Descent →](../02-Gradient-Descent/)*


In [ ]:
print("=" * 55)
print("  §01 Exercises: All 8 exercises complete")
print("=" * 55)
exercises = [
    ("1 ★",   "Convex Set Membership"),
    ("2 ★",   "Convex Function Verification"),
    ("3 ★★",  "Strong Convexity & Condition Number"),
    ("4 ★★",  "LP Feasibility and Duality"),
    ("5 ★★",  "Proximal Operator and ISTA"),
    ("6 ★★",  "Fenchel Conjugate"),
    ("7 ★★★", "SVM Dual and KKT Conditions"),
    ("8 ★★★", "Elastic Net Regularisation"),
]
for num, name in exercises:
    print(f"  Ex {num:5}: {name}")
